# TP 2 · Titanic — Identification et création des bonnes variables

**Jour 2 — chapitre 03 : Feature engineering**

## Mise en situation

Sur la base des features nettoyées ce matin (notebook précédent), il s'agit maintenant de
sélectionner et d'enrichir les variables avant la modélisation.


In [1]:
import pandas as pd
import seaborn as sns

titanic_raw = sns.load_dataset("titanic")
titanic = titanic_raw.rename(columns={
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked",
    "who": "Who",
})
titanic = titanic[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
titanic.head()

# On reprend le nettoyage du TP precedent (imputation Age / Embarked)
age_median_par_groupe = titanic.groupby(["Sex", "Pclass"])["Age"].transform("median")
titanic["Age"] = titanic["Age"].fillna(age_median_par_groupe).fillna(titanic["Age"].median())
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])
titanic.isnull().sum()


Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

## Étape 1 — Encoder les variables catégorielles (Sex, Embarked)

**Question de réflexion :** `Sex` et `Embarked` sont des catégories sans ordre naturel.
Quelle technique d'encodage utiliser, et pourquoi pas un simple encodage numérique
(0, 1, 2) ?


In [2]:
titanic_encoded = pd.get_dummies(titanic, columns=["Sex", "Embarked"], drop_first=True)
titanic_encoded.head()


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


**Ce qu'on observe** : le one-hot encoding crée une colonne binaire par catégorie (moins
une, avec `drop_first=True`, pour éviter la redondance). Un simple label encoding (0, 1, 2)
aurait introduit un ordre artificiel entre des catégories qui n'en ont pas.


## Étape 2 — Créer la variable FamilySize et la variable IsAlone

**Question de réflexion :** comment combiner `SibSp` (frères/soeurs/conjoints à bord) et
`Parch` (parents/enfants à bord) en une seule variable pertinente ? Et comment en déduire
si le passager voyageait seul ?


In [3]:
titanic_encoded["FamilySize"] = titanic_encoded["SibSp"] + titanic_encoded["Parch"] + 1
titanic_encoded["IsAlone"] = (titanic_encoded["FamilySize"] == 1).astype(int)

titanic_encoded[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()


,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


**Ce qu'on observe** : `FamilySize` compte le passager lui-même (+1), plus ses proches à
bord. `IsAlone` vaut 1 quand `FamilySize` vaut exactement 1. Ces deux variables sont des
exemples typiques de features « combinées », créées à partir de variables existantes.


## Étape 3 — Extraire le titre social depuis le nom

**Question de réflexion :** le dataset seaborn ne fournit pas la colonne `Name`. Si vous
aviez le vrai `train.csv` Kaggle, comment extrairiez-vous un titre (Mr, Mrs, Miss, Master)
depuis une chaîne de caractères comme `"Braund, Mr. Owen Harris"` ? Et en quoi ce titre
est-il une feature potentiellement plus informative que l'âge brut lui-même ?


In [4]:
# Demonstration sur des exemples de noms, au format Kaggle ("Nom de famille, Titre. Prenom")
exemples_noms = pd.Series([
    "Braund, Mr. Owen Harris",
    "Cumings, Mrs. John Bradley (Florence Briggs Thayer)",
    "Heikkinen, Miss. Laina",
    "Palsson, Master. Gosta Leonard",
])

titres_extraits = exemples_noms.str.extract(r" ([A-Za-z]+)\.")
pd.DataFrame({"Name": exemples_noms, "Title": titres_extraits[0]})


,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Palsson, Master. Gosta Leonard",Master


**Ce qu'on observe** : `str.extract` avec une expression régulière isole le mot situé
juste avant le point, qui correspond au titre social. Ce titre est un excellent proxy :
il encode à la fois l'âge approximatif (Master désigne un jeune garçon), le sexe et parfois
le statut social (Mrs vs Miss), en une seule variable catégorielle facile à encoder.

Sur le vrai dataset Kaggle : `titanic["Title"] = titanic["Name"].str.extract(r" ([A-Za-z]+)\.")`,
suivi d'un regroupement des titres rares (Dr, Rev, Col...) dans une catégorie `"Rare"`.


## Étape 4 — Sélectionner un premier jeu de features candidates

**Question de réflexion :** parmi toutes les variables désormais disponibles, lesquelles
retenir pour le prochain modèle ? Faut-il déjà tout inclure ?


In [5]:
features_candidates = [
    "Pclass", "Age", "Fare", "FamilySize", "IsAlone",
    "Sex_male",
]
features_candidates = [c for c in features_candidates if c in titanic_encoded.columns]

X = titanic_encoded[features_candidates]
y = titanic_encoded["Survived"]

X.head()


,Pclass,Age,Fare,FamilySize,IsAlone,Sex_male
0,3,22.0,7.2500,2,0,True
1,1,38.0,71.2833,2,0,False
2,3,26.0,7.9250,1,1,False
3,1,35.0,53.1000,2,0,False
4,3,35.0,8.0500,1,1,True


**Ce qu'on observe** : on part d'un jeu de features raisonnable, ni trop pauvre (on garde
les variables identifiées comme discriminantes en jour 1 : sexe, classe), ni pléthorique.
La sélection plus rigoureuse (filtre, wrapper, embarqué, vue en cours) pourra affiner ce
choix ensuite. Le prochain notebook entraîne un modèle sur ces features et vérifie s'il
fait mieux que le premier modèle du notebook 01, construit sans `Age` ni variables créées.
